In [ ]:
from google.colab import files

uploaded = files.upload()  # Upload your video file

Saving yesss.mp4 to yesss.mp4


In [ ]:
!pip install ultralytics opencv-python numpy supervision
!pip install lap deep-sort-realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.3/949.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLOv8 model (pre-trained on COCO)
model = YOLO("yolov8n.pt")

# Initialize DeepSORT tracker
tracker = DeepSort(max_age=50)

# Video paths
input_video = "/content/yesss.mp4"
output_video = "/content/output_wrong_way.mp4"

# Open video file
cap = cv2.VideoCapture(input_video)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

# Dictionary to store vehicle movement history
track_history = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 on the frame
    results = model(frame)
    detections = []

    for result in results:
        for box in result.boxes.data:
            x1, y1, x2, y2, conf, cls = box.tolist()
            class_id = int(cls)
            if class_id in [2, 3, 5, 7]:  # Car, Motorcycle, Bus, Truck
                detections.append(([x1, y1, x2, y2], conf, class_id))

    # Track objects with DeepSORT
    tracked_objects = tracker.update_tracks(detections, frame=frame)

    for track in tracked_objects:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        bbox = track.to_ltwh()
        x, y, w, h = map(int, bbox)

        # Store movement history
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append((x + w // 2, y + h // 2))

        # Check for wrong-way movement
        if len(track_history[track_id]) > 5:
            start_x, start_y = track_history[track_id][0]
            end_x, end_y = track_history[track_id][-1]

            if end_y < start_y:  # Moving opposite to expected direction
                color = (0, 0, 255)  # Red for wrong-way
                text = "WRONG WAY!"
                # Draw bounding box and text only if wrong-way detected
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
                cv2.putText(frame, f"ID {track_id}: {text}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Write frame to output video
    out.write(frame)

cap.release()
out.release()

# Download the output video
from google.colab import files
files.download(output_video)



0: 384x640 2 cars, 174.9ms
Speed: 3.3ms preprocess, 174.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 157.3ms
Speed: 3.0ms preprocess, 157.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 150.1ms
Speed: 4.8ms preprocess, 150.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 train, 148.2ms
Speed: 3.5ms preprocess, 148.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 153.9ms
Speed: 4.1ms preprocess, 153.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 145.3ms
Speed: 4.3ms preprocess, 145.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 147.9ms
Speed: 3.4ms preprocess, 147.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 152.9ms
Speed: 4.8ms preprocess, 152.9ms inference, 1.3ms postprocess per image at shape (

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort*
import torch

# Load YOLOv8 Model (Use ONNX version to avoid watermark)
model = YOLO("yolov8n.pt")  # Try "yolov8n.onnx" if watermark persists

# Initialize DeepSORT Tracker
tracker = DeepSort(max_age=50)

# Video Paths
input_video = "/content/yess.mp4"
output_video = "/content/output_wrong_way.mp4"

# Open Video File
cap = cv2.VideoCapture(input_video)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

# Track History
track_history = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 on Frame (Disable CLIFFER Messages)
    results = model(frame, verbose=False)  # Silent mode

    detections = []
    for result in results:
        for box in result.boxes.data:
            x1, y1, x2, y2, conf, cls = box.tolist()
            class_id = int(cls)
            if class_id in [2, 3, 5, 7]:  # Car, Motorcycle, Bus, Truck
                detections.append(([x1, y1, x2, y2], conf, class_id))

    # Track Objects
    tracked_objects = tracker.update_tracks(detections, frame=frame)

    for track in tracked_objects:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        bbox = track.to_ltwh()
        x, y, w, h = map(int, bbox)

        # Store Movement History
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append((x + w // 2, y + h // 2))

        # Wrong-Way Detection (Better Accuracy)
        if len(track_history[track_id]) > 10:
            start_x, start_y = track_history[track_id][0]
            end_x, end_y = track_history[track_id][-1]

            if end_y < start_y:  # Moving in the opposite direction
                color = (0, 0, 255)  # Red for Wrong Way
                text = "WRONG WAY!"
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 3)
                cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # Save Frame to Output Video
    out.write(frame)

cap.release()
out.release()

# Download Output Video
from google.colab import files
files.download(output_video)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from google.colab import files

# Load YOLOv8 Model (Try using "yolov8n.onnx" to avoid watermark)
model = YOLO("yolov8n.pt")

# Initialize DeepSORT Tracker
tracker = DeepSort(max_age=50)

# Optical Flow Parameters (Lucas-Kanade)
lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Open Video File
input_video = "/content/yess.mp4"
output_video = "/content/output_wrong_way.mp4"
cap = cv2.VideoCapture(input_video)

# Get video properties
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

# Variables for Optical Flow
prev_gray = None
track_history = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to Grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Run YOLOv8 on Frame
    results = model(frame, verbose=False)

    detections = []
    for result in results:
        for box in result.boxes.data:
            x1, y1, x2, y2, conf, cls = box.tolist()
            class_id = int(cls)
            if class_id in [2, 3, 5, 7]:  # Car, Motorcycle, Bus, Truck
                detections.append(([x1, y1, x2, y2], conf, class_id))

    # Track Objects
    tracked_objects = tracker.update_tracks(detections, frame=frame)

    for track in tracked_objects:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        bbox = track.to_ltwh()
        x, y, w, h = map(int, bbox)
        center = np.array([[x + w // 2, y + h // 2]], dtype=np.float32)

        # Store Movement History
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append(center)

        # Optical Flow for Direction Detection
        if prev_gray is not None and len(track_history[track_id]) > 5:
            prev_center = track_history[track_id][-5]  # Compare with 5 frames before
            new_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_center, center, **lk_params)
            if new_pts is not None and status[0][0] == 1:
                dx = new_pts[0][0] - prev_center[0][0]
                dy = new_pts[0][1] - prev_center[0][1]

                if dy < -2:  # Moving upwards (wrong way)
                    color = (0, 0, 255)  # Red for Wrong Way
                    text = "WRONG WAY!"
                    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 3)
                    cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    prev_gray = gray  # Update previous frame
    out.write(frame)  # Save frame to output video

cap.release()
out.release()

# Download Output Video
files.download(output_video)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>